# Grassmannian symmetry search

Use the **SageMath** kernel. Run this notebook from the repository folder. Results are quiet by default; add `verbose=True` for progress. No intermediate filenames are needed for new calculations.

In [ ]:
from symmetry_search import search, load_hypersurface, load_search

## Your completed A7 example
Load the existing equation and reuse its completed smoothness certificate.

In [ ]:
X = load_hypersurface("output/a7-equation.json")
X.f

In [ ]:
result = X.check_smoothness(prime=113)
result

In [ ]:
result.is_smooth  # True, False, or None (inconclusive)

## A fresh search
Search by k, n, d. Here we restrict to A7; omit `group` for the bounded catalogue search. Each row is a representation and scalar character, not just a group.

In [ ]:
results = search(3, 10, 1, group="A7")
results

Select a row and construct its equation. This cell performs a fresh exact calculation. `basis_tries=0` uses the original and known certified bases; the default also searches cyclic bases.

In [ ]:
Y = results[0].construct(basis_tries=0)
Y.f

The equation is an ordinary Sage polynomial in the Plücker variables. Coordinate indices start at zero.

In [ ]:
Y.p(0, 1, 7), Y.f.degree(), Y.change_of_basis.dimensions()

In [ ]:
Y.invariant_basis

A bounded smoothness pilot returns `incomplete`; repeat without `max_charts` to finish. Progress is retained automatically while the object is alive.

In [ ]:
pilot = Y.check_smoothness(prime=113, max_charts=2)
pilot

## Other groups and families
`simple=True` selects simple groups, `simple=False` non-simple groups, and `simple=None` both. Irreducibles are the default; add `reducible=True` to include sums.

In [ ]:
small = search(1, 2, 3, group="small:6:1", simple=False)
small

In [ ]:
Z = small[1].construct()
Z.f

In [ ]:
Z.check_smoothness()  # characteristic zero by default

For a higher-dimensional invariant space, use `.construct(random_section=True, seed=17)` or `.construct(coefficients=[1,2])`. To keep a result across sessions, call `.save("output/my-equation.json")`; load it with `load_hypersurface` next time. Saved checks keep their full evidence in sidecar files.

For all options, use `help(search)`, `help(results[0].construct)`, and `help(X.check_smoothness)`.

## Tests and benchmarks
Run correctness tests before and after changing an algorithm. The quick benchmark runs 15 chart cases in fresh workers (three repetitions each); it does not recheck all 120 A7 charts. Use `profile="full"` for that slower run.

In [ ]:
from symmetry_search.testing import run_tests
tests = run_tests()
assert tests.wasSuccessful()

In [ ]:
from symmetry_search.benchmark import run_benchmarks, compare_reports
baseline = run_benchmarks(output="output/my-symmetry-baseline.json")
baseline["passed"]

After modifying the code, restart the Sage kernel, rerun the tests, load your saved baseline with `json.load`, and run `trial = run_benchmarks(output="output/symmetry-trial.json")`. Then `compare_reports(trial, baseline)` reports per-case speedups. Restarting ensures the benchmark controller and tests also use your edited code. See `symmetry_search/README.md` for the saved reference run and compatibility requirements.

## Optimized smoothness
Restart the Sage kernel after updating. Worker reuse and sparse minor construction are now the defaults. Optional settings below reuse any existing certificate; use `restart=True` only when you deliberately want to recompute.

In [ ]:
X.check_smoothness(prime=113, algorithm="slimgb", symmetry=True)

The end-to-end benchmark compares fresh workers, reusable workers, parallel workers, Gröbner algorithms, and contraction equations using separate temporary checkpoints. `max_charts=None` runs the complete A7 check and also tests symmetry reduction; it takes longer because the reference method launches 120 separate Sage processes.

In [ ]:
from symmetry_search.pipeline_benchmark import run_pipeline_benchmarks
trial = run_pipeline_benchmarks(repeats=3, max_charts=12, output="output/symmetry-pipeline-trial.json")
[(v["name"], v["median_wall_seconds"], v.get("speedup")) for v in trial["variants"]]

## Fast quadratic construction
Restart the Sage kernel after updating. All degree-two candidates automatically use exact Reynolds forms and Plücker straightening, for general `Gr(k,n)`. The invariant-space dimension is certified against the character calculation; resource bounds still apply. This example performs a fresh construction.


In [ ]:
quadrics = search(3, 10, 2, group="A6")
Q = quadrics[0].construct()
len(Q.invariant_basis)  # 25


In [ ]:
Q.f


The same completed example is saved for immediate loading. Constructing an invariant equation does not establish smoothness. Use `random_section=True` to choose a reproducible combination of the invariant basis; sparse choices can be singular.


In [ ]:
Q = load_hypersurface("output/symmetry-examples/a6-gr310-quadric.json")
Q.data["quadratic_construction"]


## The A5 cubic that stalled
The coordinate-plane precheck now finds an exact characteristic-zero singular point before starting a Gröbner computation. Other difficult cases can use a bounded pilot such as `seconds=10, max_charts=1, verbose=True`; the timeout is per chart.


In [ ]:
C = load_hypersurface("output/symmetry-examples/a5-gr25-cubic.json")
C.check_smoothness(algorithm="slimgb", symmetry=True)


## Find a smooth example with faithful action
This tries candidate rows in order and reuses each invariant basis for several equations. Only certified success is returned; `None` means the chosen budget did not find a certificate. The timeout is per smoothness chart.


In [ ]:
results = search(3, 10, 2)
X = results.find_smooth(sections_per_candidate=5, prime=113, seconds=10,
                        algorithm="slimgb", symmetry=True, verbose=True)
if X is not None:
    display(X.f, X.last_smoothness, X.last_faithfulness)


In [ ]:
results.last_smooth_search
